In [2]:
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))


from src.dataset import (
    load_labels,
    make_label_mapping,
    make_transforms,
    SignLanguageDataset,
    stratified_split,
)
from src.models import SimpleMLP
from src.train_utils import evaluate

DATA_ROOT = PROJECT_ROOT / "sign_lang_train" / "sign_lang_train"

df = load_labels(DATA_ROOT)
label_to_idx, idx_to_label = make_label_mapping(df["label"])

train_df, val_df, test_df = stratified_split(
    df,
    test_size=0.2,
    val_size=0.2,
    random_state=42,
)

eval_transform = make_transforms(image_size=64, augment=False)

train_dataset = SignLanguageDataset(
    train_df,
    label_to_idx=label_to_idx,
    transform=eval_transform,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=False,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SimpleMLP(
    input_size=64 * 64,
    hidden_size=512,
    num_classes=len(label_to_idx),
).to(device)

model.load_state_dict(
    torch.load(PROJECT_ROOT / "saved_models" / "mlp_baseline_8epochs.pt",
               map_location=device)
)

criterion = nn.CrossEntropyLoss()

train_result = evaluate(
    model=model,
    loader=train_loader,
    criterion=criterion,
    device=device,
)

print("Train accuracy:", train_result["accuracy"])
print("Train loss:", train_result["loss"])
print("Train macro-F1:", train_result["macro_f1"])

Train accuracy: 0.7126715092816788
Train loss: 0.8965391489456891
Train macro-F1: 0.5799513432860174
